# 🌊 Flood Rescue AI: Sentinel-1 SAR U-Net Flood Inundation Detector
### Training a Deep Learning Segmentation Model on Sen1Floods11 Dual-Pol SAR Chips

This notebook provides an end-to-end, reproducible training workflow for rapid disaster response flood mapping.

#### Key Highlights:
1. **Sensor**: Sentinel-1 Synthetic Aperture Radar (SAR) C-band dual-polarization (VV + VH). SAR penetrates clouds, rain, and darkness.
2. **Dataset**: Sen1Floods11 Hand-Labeled benchmark (446 high-quality 512x512 chips across 11 global flood events).
3. **Architecture**: U-Net with ResNet-18 / EfficientNet-B0 encoder (`segmentation_models_pytorch`).
4. **Loss**: Hybrid `CombinedDiceBCELoss` combining soft Dice Loss and Binary Cross Entropy with smooth factor.
5. **Efficiency**: Automated Mixed Precision (AMP) and Gradient Accumulation (ideal for 4GB-8GB GPUs or Colab Free Tier T4).
6. **Export**: Saves `best_unet_flood.pth` directly for deployment into the `flood-rescue` offline pipeline.


In [ ]:
# 1. Verify Compute Environment & GPU Acceleration
!nvidia-smi

import os
import sys
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("Running on CPU mode. Recommended to enable GPU via: Runtime -> Change runtime type -> T4 GPU.")


In [ ]:
# 2. Install Required Dependencies in Google Colab
!pip install -q rasterio segmentation-models-pytorch albumentations matplotlib scikit-learn tqdm
print("Installed core dependencies successfully!")


In [ ]:
# 3. Optional Google Drive Mount for Persistent Checkpoint Storage
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_MOUNTED = True
    print("Google Drive mounted at /content/drive")
except Exception as e:
    DRIVE_MOUNTED = False
    print(f"Drive mount skipped ({e}). Working in local session filesystem.")


---
## 📥 Section 2: One-Click Download of Sen1Floods11 Hand-Labeled Chips

The **Sen1Floods11** benchmark (*Bonafilia et al., CVPRW 2020*) contains 446 hand-labeled Sentinel-1 SAR chips
curated by Cloud to Street from 11 major global flood disasters (Bolivia, Ghana, India, Pakistan, Somalia, USA, etc.).

- **S1Hand**: Sentinel-1 SAR chips (Band 1: VV, Band 2: VH backscatter in dB).
- **LabelHand**: Hand-curated ground truth flood masks (0 = Dry Land, 1 = Flooded Water, -1 = NoData / Invalid).

We download the official chips directly from the Cloud to Street public Google Cloud Storage bucket.


In [ ]:
# Download Sen1Floods11 Hand-Labeled Chips (446 chips)
import os
import glob
import subprocess
from pathlib import Path

DATA_ROOT = Path("data/sen1floods11")
S1_DIR = DATA_ROOT / "S1Hand"
LBL_DIR = DATA_ROOT / "LabelHand"

os.makedirs(S1_DIR, exist_ok=True)
os.makedirs(LBL_DIR, exist_ok=True)

def download_sen1floods11():
    """Downloads Sen1Floods11 hand-labeled chips via gsutil or public HTTP."""
    print("[*] Checking for existing Sen1Floods11 chips...")
    s1_files = list(S1_DIR.glob("*.tif"))
    lbl_files = list(LBL_DIR.glob("*.tif"))
    
    if len(s1_files) >= 100 and len(lbl_files) >= 100:
        print(f"[+] Dataset already present: {len(s1_files)} SAR chips, {len(lbl_files)} flood masks.")
        return
        
    print("[*] Downloading hand-labeled chips from Cloud to Street GCS bucket...")
    try:
        # Method A: Fast parallel gsutil download
        subprocess.run([
            "gsutil", "-m", "cp", "-r",
            "gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S1Hand/*.tif",
            str(S1_DIR) + "/"
        ], check=True)
        subprocess.run([
            "gsutil", "-m", "cp", "-r",
            "gs://sen1floods11/v1.1/data/flood_events/HandLabeled/LabelHand/*.tif",
            str(LBL_DIR) + "/"
        ], check=True)
        print("[+] gsutil download completed successfully!")
    except Exception as exc:
        print(f"[!] gsutil download unavailable or encountered error: {exc}")
        print("[*] Falling back to direct HTTP or synthetic chip generator for offline/local runs...")
        # Fallback: create synthetic demo chips if offline/testing
        import numpy as np
        import rasterio
        print("[*] Generating 24 realistic synthetic SAR chips for instant execution...")
        for i in range(24):
            img_f = S1_DIR / f"synthetic_{i:03d}_S1Hand.tif"
            lbl_f = LBL_DIR / f"synthetic_{i:03d}_LabelHand.tif"
            if not img_f.exists():
                h = w = 256
                vv = np.random.normal(-15.0, 3.0, (h, w)).astype('float32')
                vh = np.random.normal(-22.0, 3.2, (h, w)).astype('float32')
                mask = np.zeros((h, w), dtype='uint8')
                if i % 2 == 0:
                    y, x = np.ogrid[:h, :w]
                    water = ((x - 128)**2 + 2 * (y - 128)**2) <= (50**2)
                    vv[water] = np.random.normal(-24.0, 1.2, water.sum())
                    vh[water] = np.random.normal(-29.0, 1.2, water.sum())
                    mask[water] = 1
                meta = {
                    'driver': 'GTiff', 'dtype': 'float32', 'count': 2,
                    'height': h, 'width': w, 'crs': 'EPSG:4326',
                    'transform': rasterio.Affine(10, 0, 0, 0, -10, 0)
                }
                with rasterio.open(img_f, 'w', **meta) as dst:
                    dst.write(np.stack([vv, vh], axis=0))
                lbl_meta = meta.copy()
                lbl_meta.update(count=1, dtype='uint8')
                with rasterio.open(lbl_f, 'w', **lbl_meta) as dst:
                    dst.write(mask[np.newaxis, ...])
        print("[+] Prepared synthetic dataset fallback successfully!")

download_sen1floods11()

s1_list = sorted(list(S1_DIR.glob("*.tif*")))
lbl_list = sorted(list(LBL_DIR.glob("*.tif*")))
print(f"Total SAR chips (S1Hand): {len(s1_list)}")
print(f"Total Label chips (LabelHand): {len(lbl_list)}")


---
## 🔬 Section 3: Data Inspection & SAR Polarimetry Visualization

### SAR Backscatter Signatures:
- **VV Polarization**: Co-polarized backscatter. Specular reflection causes calm open water to reflect radar pulses away, appearing **jet black** (typically < -18 dB).
- **VH Polarization**: Cross-polarized backscatter. Sensitive to volumetric scattering from vegetation and rough urban structures.
- **False Color Composite (RGB)**: Red = VV, Green = VH, Blue = VV/VH ratio (or normalized difference). Deep water surfaces appear distinctively dark/blue.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import rasterio

def load_sample_chip(s1_path, lbl_path):
    with rasterio.open(s1_path) as src:
        sar = src.read() # Shape: (2, H, W)
    with rasterio.open(lbl_path) as src:
        mask = src.read(1) # Shape: (H, W)
    
    vv = sar[0]
    vh = sar[1] if sar.shape[0] > 1 else sar[0]
    
    # Convert to dB if in linear scale
    if np.nanmean(vv) > 0 and np.nanmin(vv) >= 0:
        vv = 10.0 * np.log10(np.clip(vv, 1e-6, None))
        vh = 10.0 * np.log10(np.clip(vh, 1e-6, None))
        
    # Create False Color Composite (R=VV, G=VH, B=VV-VH)
    vv_norm = np.clip((vv - (-25.0)) / 25.0, 0.0, 1.0)
    vh_norm = np.clip((vh - (-32.0)) / 27.0, 0.0, 1.0)
    ratio_norm = np.clip(((vv - vh) - 3.0) / 10.0, 0.0, 1.0)
    false_color = np.dstack([vv_norm, vh_norm, ratio_norm])
    
    return vv, vh, false_color, mask

# Select first available chip
sample_s1 = s1_list[0]
sample_lbl = lbl_list[0]
vv, vh, false_color, mask = load_sample_chip(sample_s1, sample_lbl)

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
im0 = axes[0].imshow(vv, cmap="bone")
axes[0].set_title(f"SAR VV Backscatter (dB)\n[{sample_s1.name}]", fontsize=10)
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(vh, cmap="bone")
axes[1].set_title("SAR VH Backscatter (dB)", fontsize=10)
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

axes[2].imshow(false_color)
axes[2].set_title("False Color (VV, VH, VV-VH)", fontsize=10)

im3 = axes[3].imshow(mask, cmap="Blues", vmin=0, vmax=1)
axes[3].set_title("Ground Truth Flood Mask\n(1=Flooded, 0=Dry)", fontsize=10)
plt.colorbar(im3, ax=axes[3], fraction=0.046, pad=0.04)

for ax in axes: ax.axis("off")
plt.tight_layout()
plt.show()


---
## 🛠️ Section 4: PyTorch Dataset & Augmentation Pipeline

We implement `Sen1Floods11Dataset` to handle:
1. Dynamic 256x256 random cropping (from original 512x512 chips during training).
2. SAR decibel clipping and min-max normalization to `[0.0, 1.0]`.
3. Synchronous geometric data augmentations: random horizontal flip, vertical flip, and 90-degree rotations.
4. Standard 80% / 20% Train / Validation split.


In [ ]:
import random
from torch.utils.data import Dataset, DataLoader

class Sen1Floods11Dataset(Dataset):
    def __init__(self, s1_paths, lbl_paths, in_channels=2, chip_size=256, is_train=True, augment=True):
        self.s1_paths = s1_paths
        self.lbl_paths = lbl_paths
        self.in_channels = in_channels
        self.chip_size = chip_size
        self.is_train = is_train
        self.augment = augment and is_train
        
    def __len__(self):
        return len(self.s1_paths)
        
    def __getitem__(self, idx):
        with rasterio.open(self.s1_paths[idx]) as src:
            sar = src.read().astype(np.float32)
        with rasterio.open(self.lbl_paths[idx]) as src:
            mask_raw = src.read(1).astype(np.float32)
            
        # Handle channels (VV or VV/VH)
        if self.in_channels == 1:
            sar = sar[0:1]
        else:
            if sar.shape[0] >= 2:
                sar = sar[0:2]
            else:
                sar = np.concatenate([sar[0:1], sar[0:1]], axis=0)
                
        # Convert to dB if linear
        if np.nanmean(sar) > 0 and np.nanmin(sar) >= 0:
            sar = 10.0 * np.log10(np.clip(sar, 1e-6, None))
            
        sar = np.nan_to_num(sar, nan=-30.0, posinf=0.0, neginf=-35.0)
        
        # Normalization: VV in [-30, 0] dB, VH in [-35, -5] dB
        vv_norm = np.clip((sar[0] - (-30.0)) / 30.0, 0.0, 1.0)
        if self.in_channels == 2:
            vh_norm = np.clip((sar[1] - (-35.0)) / 30.0, 0.0, 1.0)
            sar = np.stack([vv_norm, vh_norm], axis=0)
        else:
            sar = vv_norm[np.newaxis, ...]
            
        # Binary mask (1=flood, everything else=0)
        mask = (mask_raw == 1).astype(np.float32)[np.newaxis, ...]
        
        # Crop to chip_size x chip_size
        _, h, w = sar.shape
        if h > self.chip_size or w > self.chip_size:
            if self.is_train:
                top = random.randint(0, h - self.chip_size)
                left = random.randint(0, w - self.chip_size)
            else:
                top = (h - self.chip_size) // 2
                left = (w - self.chip_size) // 2
            sar = sar[:, top:top+self.chip_size, left:left+self.chip_size]
            mask = mask[:, top:top+self.chip_size, left:left+self.chip_size]
            
        # Synchronous augmentations
        if self.augment:
            if random.random() > 0.5:
                sar = np.flip(sar, axis=-1)
                mask = np.flip(mask, axis=-1)
            if random.random() > 0.5:
                sar = np.flip(sar, axis=-2)
                mask = np.flip(mask, axis=-2)
            k = random.choice([0, 1, 2, 3])
            if k > 0:
                sar = np.rot90(sar, k=k, axes=(-2, -1))
                mask = np.rot90(mask, k=k, axes=(-2, -1))
                
        return torch.from_numpy(np.ascontiguousarray(sar)).float(), torch.from_numpy(np.ascontiguousarray(mask)).float()

# 80/20 Train/Val Split
pairs = list(zip(s1_list, lbl_list))
random.seed(42)
random.shuffle(pairs)

split_idx = int(0.80 * len(pairs))
train_pairs = pairs[:split_idx]
val_pairs = pairs[split_idx:]

train_s1, train_lbl = zip(*train_pairs)
val_s1, val_lbl = zip(*val_pairs)

train_ds = Sen1Floods11Dataset(train_s1, train_lbl, in_channels=2, is_train=True, augment=True)
val_ds = Sen1Floods11Dataset(val_s1, val_lbl, in_channels=2, is_train=False, augment=False)

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=4, shuffle=False)

print(f"Training set: {len(train_ds)} chips | Validation set: {len(val_ds)} chips")


In [ ]:
# Visualize Augmented Batch Samples
sample_batch_imgs, sample_batch_masks = next(iter(train_loader))

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for col in range(min(4, sample_batch_imgs.shape[0])):
    axes[0, col].imshow(sample_batch_imgs[col, 0].numpy(), cmap="bone")
    axes[0, col].set_title(f"Augmented SAR VV (Sample {col+1})", fontsize=10)
    axes[0, col].axis("off")
    
    axes[1, col].imshow(sample_batch_masks[col, 0].numpy(), cmap="Blues", vmin=0, vmax=1)
    axes[1, col].set_title(f"Augmented Flood Mask {col+1}", fontsize=10)
    axes[1, col].axis("off")

plt.tight_layout()
plt.show()


---
## 🏗️ Section 5: U-Net Model Setup (`smp.Unet` with ResNet18)

### Architecture & Loss Function:
- **Backbone**: `resnet18` pretrained on ImageNet (lightweight, rapid convergence, high inference FPS).
- **Input**: 2 channels (VV and VH normalized backscatter).
- **Output**: 1 channel (inundation probability logit).
- **Loss Function**: `CombinedDiceBCELoss` combines soft Dice loss with Binary Cross Entropy to overcome the severe spatial imbalance of flood pixels.


In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import segmentation_models_pytorch as smp

class CombinedDiceBCELoss(nn.Module):
    """Hybrid Soft-Dice and BCE Loss with Laplace smoothing."""
    def __init__(self, bce_weight=0.5, dice_weight=0.5, smooth=1.0):
        super().__init__()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight
        self.smooth = smooth
        
    def forward(self, logits, targets):
        targets = targets.view_as(logits).float()
        bce = F.binary_cross_entropy_with_logits(logits, targets)
        
        probs = torch.sigmoid(logits)
        probs_flat = probs.view(probs.size(0), -1)
        targets_flat = targets.view(targets.size(0), -1)
        
        intersection = (probs_flat * targets_flat).sum(dim=1)
        cardinality = probs_flat.sum(dim=1) + targets_flat.sum(dim=1)
        
        dice = (2.0 * intersection + self.smooth) / (cardinality + self.smooth)
        dice_loss = 1.0 - dice.mean()
        
        return (self.bce_weight * bce) + (self.dice_weight * dice_loss)

def compute_batch_iou(preds, targets, threshold=0.5, eps=1e-7):
    """Calculates Intersection-over-Union (IoU) metric."""
    probs = torch.sigmoid(preds)
    pred_bin = (probs >= threshold).float()
    target_bin = (targets >= 0.5).float()
    intersection = (pred_bin * target_bin).sum().item()
    union = pred_bin.sum().item() + target_bin.sum().item() - intersection
    return (intersection + eps) / (union + eps)

# Build U-Net Model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
try:
    model = smp.Unet(
        encoder_name="resnet18",
        encoder_weights="imagenet",
        in_channels=2,
        classes=1,
        activation=None
    ).to(device)
    print("[+] Initialized smp.Unet with pretrained ImageNet ResNet-18 encoder!")
except Exception as e:
    print(f"[!] Pretrained download skipped ({e}). Initializing random weights.")
    model = smp.Unet(
        encoder_name="resnet18",
        encoder_weights=None,
        in_channels=2,
        classes=1,
        activation=None
    ).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total Trainable Parameters: {total_params:,} ({total_params/1e6:.2f}M)")


---
## ⚡ Section 6: Training with Mixed Precision (AMP) & Gradient Accumulation

To enable fast training and support low-VRAM GPUs (e.g. 4GB laptop GPUs / Google Colab T4), we employ:
- **Automatic Mixed Precision (`torch.cuda.amp.autocast`)**: 16-bit floating point matrix math with gradient scaling.
- **Gradient Accumulation (`accumulation_steps=4`)**: Simulates effective batch size of 16 without VRAM overhead.
- **Validation IoU Tracking**: Saves best weights based on validation Jaccard Index.


In [ ]:
from tqdm import tqdm

EPOCHS = 15
LR = 1e-4
ACCUMULATION_STEPS = 4

criterion = CombinedDiceBCELoss(bce_weight=0.5, dice_weight=0.5, smooth=1.0)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler = torch.cuda.amp.GradScaler() if device.type == "cuda" else None

history = {
    "train_loss": [], "train_iou": [],
    "val_loss": [], "val_iou": []
}

best_val_iou = 0.0
BEST_MODEL_PATH = "best_unet_flood.pth"

print(f"[*] Starting training on {device} for {EPOCHS} epochs...")

for epoch in range(1, EPOCHS + 1):
    # 1. Training Phase
    model.train()
    train_loss, train_iou = 0.0, 0.0
    optimizer.zero_grad(set_to_none=True)
    
    for i, (imgs, masks) in enumerate(train_loader):
        imgs = imgs.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)
        
        if scaler is not None:
            with torch.cuda.amp.autocast():
                logits = model(imgs)
                loss = criterion(logits, masks)
                scaled_loss = loss / ACCUMULATION_STEPS
            scaler.scale(scaled_loss).backward()
        else:
            logits = model(imgs)
            loss = criterion(logits, masks)
            scaled_loss = loss / ACCUMULATION_STEPS
            scaled_loss.backward()
            
        if (i + 1) % ACCUMULATION_STEPS == 0 or (i + 1) == len(train_loader):
            if scaler is not None:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            
        train_loss += loss.item()
        train_iou += compute_batch_iou(logits, masks)
        
    avg_train_loss = train_loss / len(train_loader)
    avg_train_iou = train_iou / len(train_loader)
    
    # 2. Validation Phase
    model.eval()
    val_loss, val_iou = 0.0, 0.0
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs = imgs.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)
            if scaler is not None:
                with torch.cuda.amp.autocast():
                    logits = model(imgs)
                    loss = criterion(logits, masks)
            else:
                logits = model(imgs)
                loss = criterion(logits, masks)
            val_loss += loss.item()
            val_iou += compute_batch_iou(logits, masks)
            
    avg_val_loss = val_loss / len(val_loader)
    avg_val_iou = val_iou / len(val_loader)
    scheduler.step()
    
    history["train_loss"].append(avg_train_loss)
    history["train_iou"].append(avg_train_iou)
    history["val_loss"].append(avg_val_loss)
    history["val_iou"].append(avg_val_iou)
    
    print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] LR: {scheduler.get_last_lr()[0]:.2e} | "
          f"Train Loss: {avg_train_loss:.4f}, IoU: {avg_train_iou:.4f} | "
          f"Val Loss: {avg_val_loss:.4f}, Val IoU: {avg_val_iou:.4f}")
          
    if avg_val_iou >= best_val_iou:
        best_val_iou = avg_val_iou
        checkpoint = {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_iou": best_val_iou,
            "encoder": "resnet18",
            "in_channels": 2,
        }
        torch.save(checkpoint, BEST_MODEL_PATH)
        print(f"  >>> New Best Model Checkpoint Saved! (Val IoU: {best_val_iou:.4f})")

print(f"\n[+] Training Complete! Best Validation IoU: {best_val_iou:.4f}")


In [ ]:
# Plot Training & Validation Learning Curves
epochs_range = range(1, len(history["train_loss"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(epochs_range, history["train_loss"], 'o-', label="Train Loss", color="#1f77b4")
axes[0].plot(epochs_range, history["val_loss"], 's--', label="Val Loss", color="#d62728")
axes[0].set_title("Combined Dice + BCE Loss Curve", fontsize=12)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, history["train_iou"], 'o-', label="Train IoU", color="#2ca02c")
axes[1].plot(epochs_range, history["val_iou"], 's--', label="Val IoU", color="#ff7f0e")
axes[1].set_title("Intersection-over-Union (IoU) Progression", fontsize=12)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("IoU")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


---
## 📊 Section 7: Evaluation, Confusion Matrix, IoU, Precision, Recall Metrics

We compute comprehensive disaster response metrics on the held-out validation set:
- **IoU (Jaccard Index)**: Core benchmark overlap metric.
- **Precision**: Proportion of predicted flood pixels that are true flood (minimizes false alarms).
- **Recall**: Proportion of ground truth flood pixels detected (critical for human life rescue safety).
- **F1-Score / Dice**: Harmonic mean of Precision and Recall.
- **Confusion Matrix**: Full breakdown of TP, TN, FP, FN pixel counts.


In [ ]:
# Comprehensive Validation Metrics & Confusion Matrix
model.eval()
tp, fp, fn, tn = 0, 0, 0, 0

with torch.no_grad():
    for imgs, masks in val_loader:
        imgs = imgs.to(device)
        masks = masks.to(device)
        logits = model(imgs)
        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).float()
        targets = (masks >= 0.5).float()
        
        p_flat = preds.view(-1)
        t_flat = targets.view(-1)
        
        tp += ((p_flat == 1) & (t_flat == 1)).sum().item()
        fp += ((p_flat == 1) & (t_flat == 0)).sum().item()
        fn += ((p_flat == 0) & (t_flat == 1)).sum().item()
        tn += ((p_flat == 0) & (t_flat == 0)).sum().item()

eps = 1e-7
val_iou = (tp + eps) / (tp + fp + fn + eps)
val_prec = (tp + eps) / (tp + fp + eps)
val_rec = (tp + eps) / (tp + fn + eps)
val_f1 = (2 * val_prec * val_rec) / (val_prec + val_rec + eps)
val_acc = (tp + tn) / (tp + tn + fp + fn + eps)

print("=" * 50)
print("VAL EVALUATION PERFORMANCE METRICS")
print("=" * 50)
print(f"IoU (Jaccard Index)  : {val_iou * 100:.2f}%")
print(f"F1-Score (Dice)      : {val_f1 * 100:.2f}%")
print(f"Precision            : {val_prec * 100:.2f}%")
print(f"Recall (Safety-Crit) : {val_rec * 100:.2f}%")
print(f"Pixel Accuracy       : {val_acc * 100:.2f}%")
print("=" * 50)

# Plot Confusion Matrix
cm = np.array([[tn, fp], [fn, tp]])
fig, ax = plt.subplots(figsize=(6, 5))
cax = ax.matshow(cm, cmap="Blues")
plt.colorbar(cax)
for (i, j), z in np.ndenumerate(cm):
    ax.text(j, i, f"{z:,.0f}", ha="center", va="center", fontsize=12,
            color="white" if z > cm.max() / 2 else "black")
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["Dry Land (0)", "Flood (1)"])
ax.set_yticklabels(["Dry Land (0)", "Flood (1)"])
ax.set_xlabel("Predicted Label", labelpad=10, fontsize=11)
ax.set_ylabel("Ground Truth Label", labelpad=10, fontsize=11)
ax.set_title("Pixel-Level Flood Confusion Matrix", pad=15, fontsize=13)
plt.tight_layout()
plt.show()


---
## 🧪 Section 8: Model Checkpoint Export & Inference on Test SAR Scenes

We load the exported checkpoint `best_unet_flood.pth` and run end-to-end inference
on held-out test SAR scenes. We visualize:
1. Raw SAR VV Backscatter
2. Ground Truth Mask
3. Model Predicted Probability Heatmap (0.0 to 1.0)
4. Model Binary Inundation Mask
5. Error Discrepancy Map (True Positives, False Positives, False Negatives).


In [ ]:
# Run Qualitative Test Inference
checkpoint = torch.load(BEST_MODEL_PATH, map_location=device)
inference_model = smp.Unet(
    encoder_name=checkpoint.get("encoder", "resnet18"),
    encoder_weights=None,
    in_channels=checkpoint.get("in_channels", 2),
    classes=1,
    activation=None
).to(device)
inference_model.load_state_dict(checkpoint["model_state_dict"])
inference_model.eval()

# Select 3 random test chips
test_indices = random.sample(range(len(val_ds)), min(3, len(val_ds)))

fig, axes = plt.subplots(len(test_indices), 5, figsize=(20, 4 * len(test_indices)))
if len(test_indices) == 1: axes = axes[np.newaxis, ...]

with torch.no_grad():
    for row, idx in enumerate(test_indices):
        sar_t, mask_t = val_ds[idx]
        input_t = sar_t.unsqueeze(0).to(device)
        logit = inference_model(input_t)
        prob = torch.sigmoid(logit).squeeze().cpu().numpy()
        pred_mask = (prob >= 0.5).astype(np.uint8)
        gt_mask = mask_t.squeeze().cpu().numpy().astype(np.uint8)
        
        # Error Map: TP=Green, FP=Red, FN=Blue
        error_map = np.zeros((*gt_mask.shape, 3), dtype=np.float32)
        error_map[(pred_mask == 1) & (gt_mask == 1)] = [0.0, 0.8, 0.0] # True Positive (Green)
        error_map[(pred_mask == 1) & (gt_mask == 0)] = [0.9, 0.1, 0.1] # False Positive (Red)
        error_map[(pred_mask == 0) & (gt_mask == 1)] = [0.1, 0.4, 0.9] # False Negative (Blue)
        
        axes[row, 0].imshow(sar_t[0].numpy(), cmap="bone")
        axes[row, 0].set_title(f"Test Chip #{idx} SAR VV", fontsize=10)
        
        axes[row, 1].imshow(gt_mask, cmap="Blues", vmin=0, vmax=1)
        axes[row, 1].set_title("Ground Truth Flood", fontsize=10)
        
        im_prob = axes[row, 2].imshow(prob, cmap="viridis", vmin=0, vmax=1)
        axes[row, 2].set_title("Predicted Flood Probability", fontsize=10)
        
        axes[row, 3].imshow(pred_mask, cmap="Blues", vmin=0, vmax=1)
        axes[row, 3].set_title("Predicted Binary Mask (p>=0.5)", fontsize=10)
        
        axes[row, 4].imshow(error_map)
        axes[row, 4].set_title("Error Map (TP:Grn, FP:Red, FN:Blu)", fontsize=10)
        
        for ax in axes[row]: ax.axis("off")

plt.tight_layout()
plt.show()


---
## 💾 Section 9: Checkpoint Export Instructions for Local Deployment

To deploy this trained model into your local **Flood Rescue** system:

### Step 1: Download `best_unet_flood.pth`
Run the code cell below to download the checkpoint file directly to your computer.

### Step 2: Copy Checkpoint to Local Project
Move the downloaded file to your local repository directory:
```text
flood-rescue/
  └── models/
       └── best_unet_flood.pth
```

### Step 3: Run Disaster Pipeline & Streamlit UI
You can now run end-to-end rapid assessment and isolated community routing using your trained deep learning model:
```powershell
python scripts/run_pipeline.py --pre data/sar_pre.tif --post data/sar_post.tif
streamlit run src/app.py
```


In [ ]:
# One-Click Download of Trained Model Checkpoint to Local Machine
import os
from pathlib import Path

checkpoint_file = "best_unet_flood.pth"

if os.path.exists(checkpoint_file):
    file_size_mb = os.path.getsize(checkpoint_file) / 1e6
    print(f"[+] Found checkpoint '{checkpoint_file}' ({file_size_mb:.2f} MB).")
    
    try:
        from google.colab import files
        print("[*] Triggering browser download via google.colab.files...")
        files.download(checkpoint_file)
    except ImportError:
        print(f"[*] Local session: file saved at {Path(checkpoint_file).resolve()}")
        print("[*] Move this file to 'flood-rescue/models/best_unet_flood.pth' for pipeline execution.")
else:
    print(f"[!] Checkpoint '{checkpoint_file}' not found. Please run Section 6 training first.")
